# Curate and verify positive synthesis records

This notebook runs the same Python curation functions as the main workflow. The bundled records and molecular-weight lookup require no API key or PDF access.

Install from the repository root with `python -m pip install -e ".[curation,datasets,notebook]"`.

## Inputs and setup

Run the cells in order. The following paths are relative to `Demo/01_data_curation/`; all required files are included.

| Input | Location | Contents |
| --- | --- | --- |
| Raw extraction records, CSV | `input/mof_extraction.csv` | Original extraction fields plus `has_main_document` and `has_supporting_document` as `True`/`False` flags. |
| Molecular-weight lookup, CSV | `input/linker_molecular_weights.csv` | Two headerless columns: linker name and molecular weight. |
| Linker prime corrections, JSON | `../../data/organic_linker_info/linker_prime_corrections.json` | Included exact DOI/name pairs shared with the main curation workflow. |
| Supplied processed reference, CSV | `reference/processed_positive_supplied.csv` | A comparison table; it is not used as a curation input. |

To test another raw extraction, save a CSV with the same schema, select it in `config.json`, and set `CHECK_EXPECTED = False` below. The availability flags must reflect whether each source document was present. For extraction outputs that retain document-path columns, use [the curation guide](../../docs/curation.md) and [Python curation code](../../src/mofinder/curation/pipeline.py).

The demonstration writes intermediate tables and a before/after preview under `outputs/`, and preserves each run under `run_history/`. With the supplied inputs, `CHECK_EXPECTED = True` compares results against `expected/`. Continue with [dataset preparation](../02_dataset_preparation/mof_dataset_preparation_demo.ipynb) after data curation.

Implementation: [demonstration runner](mof_data_curation_demo.py) and [curation functions](../../src/mofinder/curation/pipeline.py). See the [source-to-code guide](../../docs/source_to_code.md) to find the original source and its corresponding Python functions.


In [1]:
from pathlib import Path
import runpy
import pandas as pd
from IPython.display import display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "mofinder").is_dir():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from within the MOFinder repository.")

DEMO = REPO / "Demo" / "01_data_curation"
CHECK_EXPECTED = True  # Set False for custom inputs or settings.


## 1. Inspect the extraction records

Document-availability flags retain the original filter without local file paths. The full records are available in `input/mof_extraction.csv`.

In [2]:
raw = pd.read_csv(DEMO / "input" / "mof_extraction.csv")
raw[["doi", "mof_name", "metal_1", "linker_1", "modulator_1", "solvent_main", "temperature_c", "time_h"]].head(12)


,doi,mof_name,metal_1,linker_1,modulator_1,solvent_main,temperature_c,time_h
0,10.1038/s41560-018-0261-6,MIP-200,ZrCl4,"3,3′,5,5′-tetracarboxydiphenylmethane",formic acid,acetic anhydride,120.0,72.0
1,10.1038/s41560-018-0261-6,MIL-100(Fe),iron powder,"benzene-1,3,5-tricarboxylic acid",hydrofluoric acid (HF),water,150.0,144.0
2,10.1038/s41560-018-0261-6,MIL-127(Fe),FeCl3·6H2O,"3,3′,5,5′-azobenzenetetracarboxylate",NaOH,isopropanol,85.0,24.0
3,10.1038/s41560-018-0261-6,PCN-222,ZrCl4,tetrakis(4-carboxyphenyl)porphyrin,benzoic acid,"N,N-diethylformamide",120.0,48.0
4,10.1038/s41560-018-0261-6,PCN-224,ZrCl4,tetrakis(4-carboxyphenyl)porphyrin,benzoic acid,"N,N-dimethylformamide",120.0,24.0
5,10.1038/s41560-018-0261-6,MIL-160(Al),AlCl3·6H2O,"2,5-furandicarboxylic acid",NaOH,water,NaN,24.0
6,10.1038/s41560-018-0261-6,MOF-808,ZrOCl2·8H2O,"benzene-1,3,5-tricarboxylic acid",formic acid,"N,N-dimethylformamide",110.0,48.0
7,10.1038/s41560-018-0261-6,PCN-777,ZrOCl2·8H2O,"4,4′,4″-s-triazine-2,4,6-triyl-tribenzoic acid",trifluoroacetic acid,"N,N-diethylformamide",120.0,12.0
8,10.1038/s41560-018-0261-6,NU-1000,ZrOCl2·8H2O,"4,4′,4″,4″′-(pyrene-1,3,6,8-tetrayl)tetrabenzo...",benzoic acid,"N,N-dimethylformamide",NaN,NaN
9,10.1038/s41560-018-0261-6,UiO-66,ZrCl4,terephthalic acid,acetic acid,"N,N-dimethylformamide",120.0,24.0


## 2. Normalize the positive synthesis records

The script normalizes reagent names, formulas, amounts, temperatures, and durations, saves the intermediate tables, and checks the final output against the bundled expected file.


In [3]:
runner = runpy.run_path(str(DEMO / "mof_data_curation_demo.py"))
run_demo = runner["run"]
summary = run_demo(check=CHECK_EXPECTED)
pd.DataFrame(summary["stages"])


Data curation: 174 input records; 146 processed records.
Output: MOFinder\Demo\01_data_curation\run_history\20260925T185719.995974Z_15409780\outputs
Expected data curation output: PASS


Run record: Demo/01_data_curation/run_history/20260925T185719.995974Z_15409780/run_record.json


,stage,rows,file
0,initial,164,mof_extraction_1.csv
1,metals,147,mof_extraction_2.csv
2,linkers,146,mof_extraction_3.csv
3,solvents,146,mof_extraction_4.csv
4,features,146,mof_extraction_5.csv
5,connectivity,146,mof_extraction_5.csv
6,descriptions,146,mof_extraction_6.csv


## 3. Verify against expected output

Rerun this cell at any time after generating the files. It reads the current files in `outputs/`, compares them with the bundled `expected/` files, and displays expected and actual row counts with **PASS** or **FAIL**. A matching row count alone is insufficient: the check also compares the file contents. A failed check stops here with an assertion error and includes the reason in the table.

These expected results apply to the supplied inputs and settings. For your own data, set `CHECK_EXPECTED = False` in the setup cell. This skips both the run-time check and the separate comparison below.


In [4]:
if CHECK_EXPECTED:
    verification = runner["verify_outputs"](DEMO / "outputs")
    checks = pd.DataFrame(verification["checks"])
    checks["result"] = checks["passed"].map({True: "PASS", False: "FAIL"})
    display(checks[["file", "expected_rows", "actual_rows", "result", "detail"]])
    assert verification["passed"], "Output verification failed; inspect the comparison above."
else:
    print("Expected-output verification skipped (CHECK_EXPECTED = False).")


,file,expected_rows,actual_rows,result,detail
0,mof_extraction_6.csv,146.0,146.0,PASS,"All columns, rows, and values match the expect..."
1,demo_summary.json,NaN,NaN,PASS,All JSON values match the expected output.


## 4. Inspect normalized conditions and durations


In [5]:
cleaned = pd.read_csv(DEMO / "outputs" / "processed_preview.csv")
cleaned.head(12)


,doi,mof_name,metal_1,metal_1_amount_value,metal_1_amount_unit,linker_1,linker_1_amount_value,linker_1_amount_unit,solvent_main,solvent_main_ml,temperature_c,time_h,time_text,M_L_ratio,metel_concnertation
0,10.1038/s41560-018-0261-6,PCN-222,ZrCl4,0.321839,mmol,tetrakis(4-carboxyphenyl)porphyrin,0.063227,mmol,"N,N-diethylformamide",8.0,120,48.0,48 h,5.09,40.0
1,10.1038/s41560-018-0261-6,PCN-224,ZrCl4,0.643677,mmol,tetrakis(4-carboxyphenyl)porphyrin,0.063227,mmol,dimethylformamide,10.0,120,24.0,24 h,10.18,64.0
2,10.1038/s41560-018-0261-6,MOF-808,ZrOCl2·8H2O,6.702920,mmol,"benzene-1,3,5-tricarboxylic acid",7.090510,mmol,dimethylformamide,128.0,110,48.0,48 h,0.95,52.0
3,10.1038/s41560-018-0261-6,UiO-66,ZrCl4,3.000000,mmol,terephthalic acid,3.000000,mmol,dimethylformamide,20.0,120,24.0,24 h,1.00,150.0
4,10.1038/s41560-018-0261-6,MIL-101(Cr),Cr(NO3)3·9H2O,1.000000,mmol,terephthalic acid,1.000000,mmol,water,4.8,220,8.0,8 h,1.00,208.0
5,10.1038/s41560-018-0261-6,MOF-801,ZrOCl2·8H2O,10.000000,mmol,fumaric acid,10.000000,mmol,dimethylformamide,32.0,130,6.0,6 h,1.00,312.0
6,10.1021/acs.cgd.6b01732,calixMOF1,Cu(NO3)2·1H2O,0.205000,mmol,tetracarboxylic meta-substituted four-wall ary...,0.411000,mmol,dimethylformamide,45.0,100,24.0,24 h,0.50,5.0
7,10.1021/acsami.5c01476,NH2-MIL-101(Fe),FeCl3·6H2O,1.000030,mmol,2-aminoterephthalic acid,1.000280,mmol,ethanol,7.0,25,12.0,12 h,1.00,143.0
8,10.1021/acsami.5c01476,NH2-MIL-101(Fe),FeCl3·6H2O,2.497290,mmol,2-aminoterephthalic acid,1.242060,mmol,dimethylformamide,15.0,110,12.0,12 h,2.01,166.0
9,10.1021/acs.cgd.6b01533,UiO-66,ZrCl4,1.362000,mmol,terephthalic acid,1.362000,mmol,dimethylformamide,30.0,120,24.0,24 h,1.00,45.0


The processed positive table can be passed to the [dataset preparation demonstration](../02_dataset_preparation/README.md). The separate `reference/` table is the supplied full-corpus processed positive slice; frequency and outlier filters are computed on this demonstration's records when regenerating `expected/`.


## Saved run record

The executed output in this notebook preserves a readable example. The checked-in [recorded runs](recorded_runs/README.md) preserve run records and output snapshots for inspection on GitHub.

Each new run also saves its own timestamped folder under `run_history/`, including its output snapshot and run record. `outputs/` contains the latest generated files; earlier runs remain in `run_history/`. Local history is excluded from Git by default.


In [6]:
print("Saved run record:", summary["run_record"])
run_verification = summary.get("verification")
print("Verification:", "Not requested" if run_verification is None else ("PASS" if run_verification["passed"] else "FAIL"))


Saved run record: Demo/01_data_curation/run_history/20260925T185719.995974Z_15409780/run_record.json
Verification: PASS
